# MDM2 Active vs Inactive Classification

This notebook builds a **binary QSAR classifier** for MDM2 compounds using the curated activity dataset and the molecular feature matrix from the potency workflow.

**Class labels**
- Inactive = 0
- Active = 1

Because the dataset is strongly imbalanced, the workflow emphasizes **Matthews correlation coefficient (MCC)**, **balanced accuracy**, and minority-class precision/recall rather than raw accuracy alone.

## Workflow
1. Load curated activity data and molecular features
2. Keep only active and inactive compounds
3. Generate Bemis–Murcko scaffolds
4. Create scaffold-aware stratified train/validation/test splits
5. Apply training-only preprocessing
6. Train class-weighted classifiers
7. Optimize the decision threshold on validation data
8. Evaluate once on the held-out test set
9. Reduce features using ExtraTrees importance
10. Tune the 95%-importance ExtraTrees model

## 1. Imports and project paths

In [4]:
from pathlib import Path
import numpy as np
import pandas as pd

from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold

from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    classification_report, confusion_matrix, f1_score, make_scorer,
    matthews_corrcoef, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold, StratifiedKFold

In [5]:
PROJECT_DIR = Path("/Users/maryta/Downloads/mdm2_Reinvent_4_final")

CURATED_FILE = PROJECT_DIR / "data" / "processed" / "mdm2_pic50_qsar_standardized.csv"
FEATURE_FILE = (
    PROJECT_DIR / "data" / "processed" / "qsar_features"
    / "mdm2_padel_1d2d_morgan_features.csv"
)

## 2. Load curated data and molecular features

The curated table contains standardized SMILES, pIC50 values, and the assigned bioactivity class.  
The feature table contains PaDEL 1D/2D descriptors together with Morgan fingerprint features.

In [6]:
df_curated = pd.read_csv(CURATED_FILE)
df_features = pd.read_csv(FEATURE_FILE)

print("Curated dataset shape:", df_curated.shape)
print("Feature matrix shape:", df_features.shape)
print("\nCurated columns:")
print(df_curated.columns.tolist())
print("\nFirst feature columns:")
print(df_features.columns[:10].tolist())

Curated dataset shape: (4146, 9)
Feature matrix shape: (4048, 3496)

Curated columns:
['molecule_chembl_id', 'canonical_smiles', 'standardized_smiles', 'IC50_nM_median', 'pIC50', 'pIC50_mean', 'pIC50_std', 'measurement_count', 'bioactivity_class']

First feature columns:
['Name', 'padel_molecule_id', 'standardized_smiles', 'pIC50', 'nAcid', 'ALogP', 'ALogp2', 'AMR', 'apol', 'naAromAtom']


## 3. Construct the binary classification dataset

Only compounds labeled **active** or **inactive** are retained. The intermediate class is removed.

In [7]:
label_df = df_curated[
    ["standardized_smiles", "pIC50", "bioactivity_class"]
].copy()

df_binary = df_features.merge(
    label_df,
    on="standardized_smiles",
    how="left",
    suffixes=("", "_label"),
)

print("After merge shape:", df_binary.shape)
print("Missing activity labels:", df_binary["bioactivity_class"].isna().sum())

df_binary = df_binary[
    df_binary["bioactivity_class"].isin(["active", "inactive"])
].copy()

print("\nBinary classification dataset shape:", df_binary.shape)
print("\nClass counts:")
print(df_binary["bioactivity_class"].value_counts())
print("\nClass percentages:")
print((df_binary["bioactivity_class"].value_counts(normalize=True) * 100).round(2))

After merge shape: (4048, 3498)
Missing activity labels: 0

Binary classification dataset shape: (3757, 3498)

Class counts:
bioactivity_class
active      3593
inactive     164
Name: count, dtype: int64

Class percentages:
bioactivity_class
active      95.63
inactive     4.37
Name: proportion, dtype: float64


## 4. Generate Bemis–Murcko scaffolds

Scaffolds are used as grouping variables during splitting to reduce structural leakage between training and evaluation sets.

In [8]:
def get_murcko_scaffold(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    scaffold = MurckoScaffold.GetScaffoldForMol(mol)
    return Chem.MolToSmiles(scaffold, canonical=True)

df_binary["murcko_scaffold"] = (
    df_binary["standardized_smiles"].apply(get_murcko_scaffold)
)

print("Missing scaffolds:", df_binary["murcko_scaffold"].isna().sum())
print("Unique scaffolds:", df_binary["murcko_scaffold"].nunique())

Missing scaffolds: 0
Unique scaffolds: 1367


## 5. Scaffold-aware stratified split

The final splitting strategy in the source notebook uses `StratifiedGroupKFold` with 20 folds:

- 14 folds → train (~70%)
- 3 folds → validation (~15%)
- 3 folds → test (~15%)

Activity class is used for stratification and Murcko scaffold for grouping.

In [9]:
df_binary = df_binary.reset_index(drop=True)

X_dummy = np.zeros(len(df_binary))
y_split = (
    df_binary["bioactivity_class"]
    .map({"inactive": 0, "active": 1})
    .to_numpy()
)
groups = df_binary["murcko_scaffold"].to_numpy()

sgkf = StratifiedGroupKFold(
    n_splits=20,
    shuffle=True,
    random_state=42,
)

fold_id = np.full(len(df_binary), -1, dtype=int)

for fold, (_, fold_indices) in enumerate(
    sgkf.split(X_dummy, y_split, groups=groups)
):
    fold_id[fold_indices] = fold

df_binary["fold"] = fold_id

train_folds = list(range(0, 14))
validation_folds = list(range(14, 17))
test_folds = list(range(17, 20))

df_binary["split"] = np.select(
    [
        df_binary["fold"].isin(train_folds),
        df_binary["fold"].isin(validation_folds),
        df_binary["fold"].isin(test_folds),
    ],
    ["train", "validation", "test"],
    default="unassigned",
)

print("Split sizes:")
print(df_binary["split"].value_counts())
print("\nClass counts:")
print(pd.crosstab(df_binary["split"], df_binary["bioactivity_class"]))
print("\nClass percentages:")
print(
    pd.crosstab(
        df_binary["split"],
        df_binary["bioactivity_class"],
        normalize="index",
    ).mul(100).round(2)
)

Split sizes:
split
train         2619
test           571
validation     567
Name: count, dtype: int64

Class counts:
bioactivity_class  active  inactive
split                              
test                  539        32
train                2509       110
validation            545        22

Class percentages:
bioactivity_class  active  inactive
split                              
test                94.40      5.60
train               95.80      4.20
validation          96.12      3.88


## 6. Confirm zero scaffold leakage

In [10]:
train_scaffolds = set(df_binary.loc[df_binary["split"] == "train", "murcko_scaffold"])
val_scaffolds = set(df_binary.loc[df_binary["split"] == "validation", "murcko_scaffold"])
test_scaffolds = set(df_binary.loc[df_binary["split"] == "test", "murcko_scaffold"])

print("Train–Validation overlap:", len(train_scaffolds & val_scaffolds))
print("Train–Test overlap:", len(train_scaffolds & test_scaffolds))
print("Validation–Test overlap:", len(val_scaffolds & test_scaffolds))

Train–Validation overlap: 0
Train–Test overlap: 0
Validation–Test overlap: 0


## 7. Prepare predictors and binary target

In [11]:
non_feature_columns = [
    "Name", "padel_molecule_id", "standardized_smiles",
    "pIC50", "pIC50_label", "bioactivity_class",
    "murcko_scaffold", "fold", "split",
]

feature_columns = [
    col for col in df_binary.columns
    if col not in non_feature_columns
]

train_df = df_binary[df_binary["split"] == "train"].copy()
val_df = df_binary[df_binary["split"] == "validation"].copy()
test_df = df_binary[df_binary["split"] == "test"].copy()

X_train = train_df[feature_columns].copy()
X_val = val_df[feature_columns].copy()
X_test = test_df[feature_columns].copy()

label_map = {"inactive": 0, "active": 1}

y_train = train_df["bioactivity_class"].map(label_map).astype(int)
y_val = val_df["bioactivity_class"].map(label_map).astype(int)
y_test = test_df["bioactivity_class"].map(label_map).astype(int)

print("Number of candidate features:", len(feature_columns))
print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:  ", X_val.shape, "y_val:", y_val.shape)
print("X_test: ", X_test.shape, "y_test:", y_test.shape)

Number of candidate features: 3492
X_train: (2619, 3492) y_train: (2619,)
X_val:   (567, 3492) y_val: (567,)
X_test:  (571, 3492) y_test: (571,)


## 8. Training-only preprocessing

The cleaned workflow keeps the final preprocessing logic from the source notebook:

- replace `±inf` with missing values
- mask clearly invalid extreme PaDEL values (`|x| > 1e100`)
- remove features with >20% missing values in training
- median-impute using training statistics
- remove zero-variance features using the training set

The fitted transformations are then applied to validation and test data.

In [12]:
EXTREME_LIMIT = 1e100

X_train_clean = X_train.replace([np.inf, -np.inf], np.nan).copy()
X_val_clean = X_val.replace([np.inf, -np.inf], np.nan).copy()
X_test_clean = X_test.replace([np.inf, -np.inf], np.nan).copy()

X_train_clean = X_train_clean.mask(X_train_clean.abs() > EXTREME_LIMIT)
X_val_clean = X_val_clean.mask(X_val_clean.abs() > EXTREME_LIMIT)
X_test_clean = X_test_clean.mask(X_test_clean.abs() > EXTREME_LIMIT)

missing_fraction_train = X_train_clean.isna().mean()
kept_columns = X_train_clean.columns[missing_fraction_train <= 0.20]

X_train_f = X_train_clean[kept_columns].copy()
X_val_f = X_val_clean[kept_columns].copy()
X_test_f = X_test_clean[kept_columns].copy()

imputer = SimpleImputer(strategy="median")

X_train_imp = imputer.fit_transform(X_train_f)
X_val_imp = imputer.transform(X_val_f)
X_test_imp = imputer.transform(X_test_f)

variance_filter = VarianceThreshold(threshold=0.0)

X_train_proc = variance_filter.fit_transform(X_train_imp)
X_val_proc = variance_filter.transform(X_val_imp)
X_test_proc = variance_filter.transform(X_test_imp)

final_feature_names = kept_columns[variance_filter.get_support()].tolist()

print("Original features:", X_train.shape[1])
print("After missing-value filter:", len(kept_columns))
print("After zero-variance removal:", len(final_feature_names))

for name, X in [
    ("TRAIN", X_train_proc),
    ("VALIDATION", X_val_proc),
    ("TEST", X_test_proc),
]:
    print(
        name,
        "NaN:", np.isnan(X).sum(),
        "Inf:", np.isinf(X).sum(),
        "Max abs:", np.max(np.abs(X)),
    )

Original features: 3492
After missing-value filter: 3393
After zero-variance removal: 3088
TRAIN NaN: 0 Inf: 0 Max abs: 149413.5330233808
VALIDATION NaN: 0 Inf: 0 Max abs: 53000010937.0
TEST NaN: 0 Inf: 0 Max abs: 81101.56871749941


## 9. Baseline class-weighted ExtraTrees classifier

Because the inactive class is rare, `class_weight="balanced"` is used.  
MCC and balanced accuracy are emphasized over plain accuracy.

In [23]:
def evaluate_classifier(y_true, y_pred, y_prob, name):
    print(f"\n{name}")
    print("-" * 50)
    print("Accuracy:", round(accuracy_score(y_true, y_pred), 4))
    print("Balanced accuracy:", round(balanced_accuracy_score(y_true, y_pred), 4))
    print("MCC:", round(matthews_corrcoef(y_true, y_pred), 4))
    print("ROC-AUC:", round(roc_auc_score(y_true, y_prob), 4))
    print("PR-AUC (active=1):", round(average_precision_score(y_true, y_prob), 4))
    print("\nConfusion matrix:")
    print(confusion_matrix(y_true, y_pred))
    print("\nClassification report:")
    print(
        classification_report(
            y_true,
            y_pred,
            target_names=["inactive", "active"],
            digits=4,
        )
    )

clf = ExtraTreesClassifier(
    n_estimators=1000,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)

clf.fit(X_train_proc, y_train)

val_prob = clf.predict_proba(X_val_proc)[:, 1]
test_prob = clf.predict_proba(X_test_proc)[:, 1]

val_pred_default = (val_prob >= 0.50).astype(int)

evaluate_classifier(
    y_val,
    val_pred_default,
    val_prob,
    "VALIDATION — DEFAULT THRESHOLD 0.50",
)


VALIDATION — DEFAULT THRESHOLD 0.50
--------------------------------------------------
Accuracy: 0.9683
Balanced accuracy: 0.7
MCC: 0.4977
ROC-AUC: 0.9241
PR-AUC (active=1): 0.9941

Confusion matrix:
[[  9  13]
 [  5 540]]

Classification report:
              precision    recall  f1-score   support

    inactive     0.6429    0.4091    0.5000        22
      active     0.9765    0.9908    0.9836       545

    accuracy                         0.9683       567
   macro avg     0.8097    0.7000    0.7418       567
weighted avg     0.9635    0.9683    0.9648       567



## 10. Optimize the decision threshold on validation data only

The source notebook scans probability thresholds from 0.05 to 0.95 and selects the one that maximizes **validation MCC**.  
The test set is not used during threshold selection.

In [14]:
thresholds = np.arange(0.05, 0.951, 0.01)
threshold_results = []

for threshold in thresholds:
    val_pred_t = (val_prob >= threshold).astype(int)

    threshold_results.append({
        "threshold": threshold,
        "MCC": matthews_corrcoef(y_val, val_pred_t),
        "balanced_accuracy":
            balanced_accuracy_score(y_val, val_pred_t),
    })

threshold_results = pd.DataFrame(threshold_results)

best_row = (
    threshold_results
    .sort_values("MCC", ascending=False)
    .iloc[0]
)

best_threshold = float(best_row["threshold"])

print("Best validation threshold:", round(best_threshold, 2))
print("Validation MCC:", round(best_row["MCC"], 4))
print(
    "Validation balanced accuracy:",
    round(best_row["balanced_accuracy"], 4),
)

val_pred_opt = (val_prob >= best_threshold).astype(int)

evaluate_classifier(
    y_val,
    val_pred_opt,
    val_prob,
    "VALIDATION — OPTIMIZED THRESHOLD",
)

Best validation threshold: 0.65
Validation MCC: 0.6708
Validation balanced accuracy: 0.8763

VALIDATION — OPTIMIZED THRESHOLD
--------------------------------------------------
Accuracy: 0.9718
Balanced accuracy: 0.8763
MCC: 0.6708
ROC-AUC: 0.9241
PR-AUC (active=1): 0.9941

Confusion matrix:
[[ 17   5]
 [ 11 534]]

Classification report:
              precision    recall  f1-score   support

    inactive     0.6071    0.7727    0.6800        22
      active     0.9907    0.9798    0.9852       545

    accuracy                         0.9718       567
   macro avg     0.7989    0.8763    0.8326       567
weighted avg     0.9758    0.9718    0.9734       567



## 11. Held-out test evaluation

The validation-selected threshold is applied unchanged to the held-out test set.

In [24]:
test_pred_opt = (test_prob >= best_threshold).astype(int)

evaluate_classifier(
    y_test,
    test_pred_opt,
    test_prob,
    "TEST — VALIDATION-SELECTED THRESHOLD",
)


TEST — VALIDATION-SELECTED THRESHOLD
--------------------------------------------------
Accuracy: 0.9702
Balanced accuracy: 0.8373
MCC: 0.7066
ROC-AUC: 0.9732
PR-AUC (active=1): 0.9976

Confusion matrix:
[[ 22  10]
 [  7 532]]

Classification report:
              precision    recall  f1-score   support

    inactive     0.7586    0.6875    0.7213        32
      active     0.9815    0.9870    0.9843       539

    accuracy                         0.9702       571
   macro avg     0.8701    0.8373    0.8528       571
weighted avg     0.9691    0.9702    0.9695       571



## 12. Benchmark class-weighted classifiers

The source notebook compares Logistic Regression, Random Forest, and ExtraTrees on validation data.  
Inactive-class precision, recall, and F1 are reported explicitly.

In [18]:
models = {
    "Logistic Regression": LogisticRegression(
        class_weight="balanced",
        max_iter=5000,
        random_state=42,
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=1000,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    ),
    "ExtraTrees": ExtraTreesClassifier(
        n_estimators=1000,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    ),
}

benchmark_rows = []

for model_name, model in models.items():
    model.fit(X_train_proc, y_train)

    pred = model.predict(X_val_proc)
    prob = model.predict_proba(X_val_proc)[:, 1]

    benchmark_rows.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_val, pred),
        "Balanced Accuracy": balanced_accuracy_score(y_val, pred),
        "MCC": matthews_corrcoef(y_val, pred),
        "ROC-AUC": roc_auc_score(y_val, prob),
        "Inactive Precision": precision_score(
            y_val, pred, pos_label=0, zero_division=0
        ),
        "Inactive Recall": recall_score(
            y_val, pred, pos_label=0, zero_division=0
        ),
        "Inactive F1": f1_score(
            y_val, pred, pos_label=0, zero_division=0
        ),
    })

benchmark_results = (
    pd.DataFrame(benchmark_rows)
    .sort_values("MCC", ascending=False)
    .reset_index(drop=True)
)

display(benchmark_results.round(4))

/Users/maryta/miniconda3/envs/mdm2_reinvent4/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 5000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=5000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,Model,Accuracy,Balanced Accuracy,MCC,ROC-AUC,Inactive Precision,Inactive Recall,Inactive F1
0,Random Forest,0.9735,0.7681,0.6073,0.9715,0.7059,0.5455,0.6154
1,Logistic Regression,0.9524,0.8226,0.5177,0.9648,0.4286,0.6818,0.5263
2,ExtraTrees,0.9683,0.7000,0.4977,0.9241,0.6429,0.4091,0.5000


## 13. ExtraTrees feature-importance reduction

Features are ranked by ExtraTrees importance and subsets retaining 95%, 97.5%, 99%, and 100% cumulative importance are compared on validation data.

In [25]:
feature_selector_model = ExtraTreesClassifier(
    n_estimators=1000,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)

feature_selector_model.fit(X_train_proc, y_train)

feature_importance_df = pd.DataFrame({
    "feature": final_feature_names,
    "importance": feature_selector_model.feature_importances_,
}).sort_values(
    "importance",
    ascending=False,
).reset_index(drop=True)

feature_importance_df["cumulative_importance"] = (
    feature_importance_df["importance"].cumsum()
)

display(feature_importance_df.head(20))

,feature,importance,cumulative_importance
0,Morgan_r2_bit_1671,0.013519,0.013519
1,Morgan_r2_bit_1855,0.010894,0.024413
2,Morgan_r2_bit_0352,0.010831,0.035244
3,Morgan_r2_bit_0378,0.010145,0.045389
4,Morgan_r2_bit_1978,0.007346,0.052735
5,SRW5,0.007320,0.060056
6,Morgan_r2_bit_0802,0.006749,0.066805
7,Morgan_r2_bit_0323,0.006291,0.073096
8,maxaaN,0.006205,0.079301
9,Morgan_r2_bit_0690,0.006024,0.085325


In [26]:
importance_levels = [0.95, 0.975, 0.99, 1.00]
threshold_grid = np.arange(0.05, 0.951, 0.01)

feature_index_map = {
    name: i for i, name in enumerate(final_feature_names)
}

reduction_results = []
selected_feature_sets = {}

for importance_level in importance_levels:

    if importance_level < 1.0:
        n_features = (
            np.searchsorted(
                feature_importance_df["cumulative_importance"].values,
                importance_level,
            )
            + 1
        )
    else:
        n_features = len(feature_importance_df)

    selected_names = (
        feature_importance_df
        .iloc[:n_features]["feature"]
        .tolist()
    )
    selected_feature_sets[importance_level] = selected_names

    selected_indices = [
        feature_index_map[name]
        for name in selected_names
    ]

    X_train_selected = X_train_proc[:, selected_indices]
    X_val_selected = X_val_proc[:, selected_indices]

    model = ExtraTreesClassifier(
        n_estimators=1000,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    )
    model.fit(X_train_selected, y_train)

    val_prob_selected = model.predict_proba(X_val_selected)[:, 1]

    best_mcc = -np.inf
    best_threshold_level = None
    best_pred = None

    for threshold in threshold_grid:
        pred = (val_prob_selected >= threshold).astype(int)
        mcc = matthews_corrcoef(y_val, pred)

        if mcc > best_mcc:
            best_mcc = mcc
            best_threshold_level = threshold
            best_pred = pred

    reduction_results.append({
        "Importance retained": importance_level,
        "Number of features": n_features,
        "Reduction %": 100 * (
            1 - n_features / len(final_feature_names)
        ),
        "Best threshold": best_threshold_level,
        "MCC": best_mcc,
        "Balanced Accuracy":
            balanced_accuracy_score(y_val, best_pred),
        "Inactive Precision":
            precision_score(
                y_val, best_pred,
                pos_label=0, zero_division=0
            ),
        "Inactive Recall":
            recall_score(
                y_val, best_pred,
                pos_label=0, zero_division=0
            ),
        "Inactive F1":
            f1_score(
                y_val, best_pred,
                pos_label=0, zero_division=0
            ),
        "ROC-AUC": roc_auc_score(
            y_val, val_prob_selected
        ),
    })

reduction_results_df = pd.DataFrame(reduction_results)
display(reduction_results_df.round(4))

,Importance retained,Number of features,Reduction %,Best threshold,MCC,Balanced Accuracy,Inactive Precision,Inactive Recall,Inactive F1,ROC-AUC
0,0.950,1356,56.0881,0.68,0.6708,0.8763,0.6071,0.7727,0.6800,0.9287
1,0.975,1542,50.0648,0.66,0.6708,0.8763,0.6071,0.7727,0.6800,0.9154
2,0.990,1747,43.4262,0.66,0.6708,0.8763,0.6071,0.7727,0.6800,0.9273
3,1.000,3088,0.0000,0.67,0.6581,0.8754,0.5862,0.7727,0.6667,0.9153


## 14. Hyperparameter tuning of the 95%-importance ExtraTrees model

The source notebook tunes `ExtraTreesClassifier` on the 95% cumulative-importance feature subset using **5-fold cross-validated MCC on training data only**.

In [22]:
selected_names_95 = selected_feature_sets[0.95]

selected_indices_95 = [
    feature_index_map[name]
    for name in selected_names_95
]

X_train_95 = X_train_proc[:, selected_indices_95]
X_val_95 = X_val_proc[:, selected_indices_95]
X_test_95 = X_test_proc[:, selected_indices_95]

base_model = ExtraTreesClassifier(
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)

param_grid = {
    "n_estimators": [500, 1000],
    "max_features": ["sqrt", 0.2, 0.5],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2, 4],
    "max_depth": [None, 20, 40],
}

mcc_scorer = make_scorer(matthews_corrcoef)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

grid_search = GridSearchCV(
    estimator=base_model,
    param_grid=param_grid,
    scoring=mcc_scorer,
    cv=cv,
    n_jobs=-1,
    verbose=2,
    return_train_score=True,
)

grid_search.fit(X_train_95, y_train)

print("Best parameters:")
print(grid_search.best_params_)

print("\nBest cross-validation MCC:")
print(round(grid_search.best_score_, 4))

best_et_model = grid_search.best_estimator_

Fitting 5 folds for each of 108 candidates, totalling 540 fits
Best parameters:
{'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 4, 'min_samples_split': 2, 'n_estimators': 1000}

Best cross-validation MCC:
0.7309


## 15. Interpretation and scope

This cleaned notebook removes duplicated and troubleshooting-only cells while preserving the final modeling path from the source.

Removed from the cleaned version:
- the earlier unsuccessful greedy scaffold split
- duplicated threshold-optimization blocks
- repeated preprocessing and ExtraTrees training cells
- debugging cells used only to diagnose invalid PaDEL values

**Important:** the source notebook ends after hyperparameter tuning of the 95%-importance ExtraTrees model. It does not show a new held-out test evaluation of that tuned model. This cleaned notebook therefore does not invent one.